# Defining a custom indicator in earthkit climate

This tutorial implements a weather-based **Fusarium head blight (FHB) conducive-days indicator**. This is a destructive fungal disease that attacks cereal crops like wheat, barley, and oats. This indicator is not available in xclim, so it is also not available in earthkit-climate, which currently only supports xclim indicators.

A day is FHB-conducive when all three conditions hold:

* mean temperature: 15 °C ≤ T ≤ 30 °C;
* relative humidity: RH ≥ 90%;
* daily precipitation: P > 0 mm.

The indicator output is the yearly number of days satisfying all three conditions. This is the definition used by the C3S2_414 contract (Operational Copernicus Climate Change Agriculture Service).

In order to implement the indicator and make it usable from earthkit climate, we can follow two paths:

a) Define the indicator as a xclim indicator, and then use the earthkit format wrapper.
b) Alternatively, avoid using xclim entirely and use earthkit functions exclusively.

While a) does have some advantages (like the automatic unit and metadata support), b) can be more compact. Both methods are similar, the main difference is that the xclim indicator requires using a Indicator class (Daily in this case), while in earthkit the compute function can be directly turned into an indicator using the @format_handler()
 wrapper as a decorator.

Once the indicator is implemented, it is advised to test it with synthetic data, as it is shown in the notebook. Finally, the user can open a pull request in github to include the indicator in earthkit climate (see https://earthkit.readthedocs.io/en/latest/development/guidelines.html). This will ensure that it is maintained and tested in the future releases of the package.

## Contents

- [1. Define the compute function](#1-define-the-compute-function)
- [2. Create the xclim Indicator](#2-create-the-xclim-indicator)
- [3. Add the earthkit format wrapper](#3-add-the-earthkit-format-wrapper)
- [4. Build deterministic sample data](#4-build-deterministic-sample-data)
- [5. Compute and verify yearly FHB-conducive days](#5-compute-and-verify-yearly-fhb-conducive-days)
- [6. Implement the same indicator without xclim](#6-implement-the-same-indicator-without-xclim)
- [7. Compare the two approaches](#7-compare-the-two-approaches)

In [4]:
from __future__ import annotations

from typing import Any

import earthkit.transforms as ekt
import numpy as np
import pandas as pd
import xarray as xr
from earthkit.utils.decorators import format_handler
from xclim.core.indicator import Daily
from xclim.core.units import convert_units_to, declare_units, rate2amount, to_agg_units

## 1. Define the compute function

The compute function owns the scientific definition. xclim is used only for unit-safe conversion, daily precipitation amount conversion, and aggregation units. The three threshold masks are combined with logical AND, so every condition must hold on a qualifying day.

In [5]:
@declare_units(
    tas="[temperature]",
    hurs="[]",
    pr="[precipitation]",
    tas_lower="[temperature]",
    tas_upper="[temperature]",
    hurs_lower="[]",
    pr_lower="[length]",
)
def fusarium_head_blight_conducive_days_index(
    tas: xr.DataArray,
    hurs: xr.DataArray,
    pr: xr.DataArray,
    tas_lower: str = "15 degC",
    tas_upper: str = "30 degC",
    hurs_lower: str = "90 %",
    pr_lower: str = "0 mm",
    freq: str = "YS",
) -> xr.DataArray:
    """Return the number of Fusarium head blight conducive days.

    A day is conducive when mean temperature is within the closed interval
    [tas_lower, tas_upper], relative humidity is at least hurs_lower, and
    daily precipitation is strictly greater than pr_lower.

    Parameters
    ----------
    tas : xarray.DataArray
        Daily mean near-surface air temperature.
    hurs : xarray.DataArray
        Daily mean relative humidity.
    pr : xarray.DataArray
        Daily precipitation flux or rate.
    tas_lower : str
        Inclusive lower temperature threshold.
    tas_upper : str
        Inclusive upper temperature threshold.
    hurs_lower : str
        Inclusive lower relative-humidity threshold.
    pr_lower : str
        Exclusive lower daily precipitation threshold.
    freq : str
        Aggregation frequency; ``YS`` produces one result per calendar year.

    Returns
    -------
    xarray.DataArray
        Number of FHB-conducive days for each period.
    """
    tas_c = convert_units_to(tas, "degC")
    hurs_pct = convert_units_to(hurs, "%")
    pr_amount = rate2amount(pr, out_units="mm")

    lower_t = convert_units_to(tas_lower, tas_c)
    upper_t = convert_units_to(tas_upper, tas_c)
    lower_rh = convert_units_to(hurs_lower, hurs_pct)
    lower_pr = convert_units_to(pr_lower, pr_amount)

    conducive = (tas_c >= lower_t) & (tas_c <= upper_t) & (hurs_pct >= lower_rh) & (pr_amount > lower_pr)

    out = ekt.resample(conducive, frequency=freq, how="sum")
    out = to_agg_units(out, tas, op="count")
    out.attrs.pop("standard_name", None)
    return out.rename("fhb_conducive_days")

## 2. Create the xclim Indicator

The xclim layer adds daily-frequency validation, missing-value handling, dataset variable lookup, and output metadata. We instantiate xclim's existing `Daily` base because this single indicator does not need a new class with shared validation behaviour. See https://xclim.readthedocs.io/en/stable/notebooks/extendxclim.html#Defining-new-indicators.

In [6]:
fhb_conducive_days_xclim = Daily(
    realm="atmos",
    identifier="fhb_conducive_days",
    compute=fusarium_head_blight_conducive_days_index,
    title="Fusarium head blight conducive days",
    abstract=("Yearly number of wet, very humid days with temperatures conducive to Fusarium head blight."),
    var_name="fhb_conducive_days",
    long_name="Fusarium head blight conducive days",
    description=(
        "Number of days where {tas_lower} <= T <= {tas_upper}, relative humidity "
        ">= {hurs_lower}, and precipitation > {pr_lower}."
    ),
    units="d",
    cell_methods="time: sum over days",
)

## 3. Add the earthkit format wrapper

`format_handler()` is a decorator factory, so decorator syntax and an explicit call are equivalent. Here we apply it directly to the callable xclim `Indicator`.

```python
earthkit_callable = format_handler()(xclim_indicator)
```

The resulting callable accepts xarray objects as well as supported earthkit inputs.

In [7]:
fusarium_head_blight_conducive_days = format_handler()(fhb_conducive_days_xclim)

## 4. Build deterministic sample data

The synthetic dataset contains two complete years of daily mean temperature, relative humidity, and precipitation. It includes qualifying days and near misses where exactly one of the three conditions fails, making the logical-AND combiner directly testable.

In [8]:
time = pd.date_range("2020-01-01", "2021-12-31", freq="D")
shape = time.size

tas = xr.DataArray(
    np.full(shape, 20.0),
    coords={"time": time},
    dims="time",
    name="tas",
    attrs={
        "units": "degC",
        "standard_name": "air_temperature",
        "cell_methods": "time: mean within days",
    },
)
hurs = xr.DataArray(
    np.full(shape, 70.0),
    coords={"time": time},
    dims="time",
    name="hurs",
    attrs={
        "units": "%",
        "standard_name": "relative_humidity",
        "cell_methods": "time: mean within days",
    },
)
pr = xr.DataArray(
    np.zeros(shape),
    coords={"time": time},
    dims="time",
    name="pr",
    attrs={
        "units": "mm/day",
        "standard_name": "precipitation_flux",
        "cell_methods": "time: mean within days",
    },
)

# Days satisfying all three FHB conditions.
fhb_dates_2020 = ["2020-06-10", "2020-06-11", "2020-06-20"]
fhb_dates_2021 = ["2021-05-15", "2021-05-16", "2021-05-17", "2021-05-30"]
fhb_dates = fhb_dates_2020 + fhb_dates_2021
hurs.loc[{"time": fhb_dates}] = 95.0
pr.loc[{"time": fhb_dates}] = 1.0

# Near misses demonstrate that the combiner is ALL.
near_miss_dates = ["2020-06-25", "2021-06-01", "2021-06-02"]
hurs.loc[{"time": near_miss_dates}] = 95.0
pr.loc[{"time": near_miss_dates}] = 1.0
tas.loc[{"time": "2020-06-25"}] = 31.0  # Temperature above the upper bound.
hurs.loc[{"time": "2021-06-01"}] = 85.0  # Humidity below the lower bound.
pr.loc[{"time": "2021-06-02"}] = 0.0  # No precipitation.

ds = xr.Dataset({"tas": tas, "hurs": hurs, "pr": pr})
ds.sel(time=fhb_dates + near_miss_dates).to_dataframe()

,tas,hurs,pr
time,,,
2020-06-10,20.0,95.0,1.0
2020-06-11,20.0,95.0,1.0
2020-06-20,20.0,95.0,1.0
2021-05-15,20.0,95.0,1.0
2021-05-16,20.0,95.0,1.0
2021-05-17,20.0,95.0,1.0
2021-05-30,20.0,95.0,1.0
2020-06-25,31.0,95.0,1.0
2021-06-01,20.0,85.0,1.0


## 5. Compute and verify yearly FHB-conducive days

Every day satisfying all three conditions is counted: three days in 2020 and four in 2021. The near misses are excluded. The assertions make these expected values part of the executable tutorial.

In [9]:
fhb_days = fusarium_head_blight_conducive_days(ds=ds)

np.testing.assert_array_equal(fhb_days.values, [3, 4])
np.testing.assert_array_equal(fhb_days.time.dt.year.values, [2020, 2021])
assert fhb_days.attrs["units"] == "d"

fhb_days.to_dataframe()

,fhb_conducive_days
time,
2020-01-01,3.0
2021-01-01,4.0


The indicator is independent of xclim's published indicator catalogue: the threshold combination and FHB interpretation are defined in our compute function, while xclim supplies reusable unit, validation, missing-data, and metadata infrastructure. Before operational use, the thresholds and relevant crop-stage period should be validated for the target cultivar, pathogen population, and region.

## 6. Implement the same indicator without xclim

The following version uses only earthkit utilities plus xarray, earthkit's native multidimensional array representation:

- `format_handler` translates supported earthkit inputs to xarray;
- `earthkit.utils.units.convert_units` normalizes both input data and unit-bearing thresholds;
- the function explicitly resolves variable names from `ds`;
- xarray performs the threshold combination;
- `earthkit.transforms.resample` performs the yearly aggregation.

Its public signature deliberately mirrors the xclim-backed earthkit wrapper, including threshold names, defaults, and `**kwargs`. For daily inputs, precipitation is normalized to `mm/day`, while the `pr_lower` amount is interpreted per daily sample.

This implementation applies a strict missing-data rule in a single resampling operation. `conducive.where(valid)` changes the daily result to missing wherever any input is missing. Then `how_kwargs={"skipna": False}` tells the yearly sum not to ignore those missing days, so the whole yearly result is missing. This replaces the less direct approach of separately resampling the validity mask and applying it with a second `where`.

In [10]:
from earthkit.utils.units import convert_units as ek_convert_units


def _resolve_dataarray(
    value: xr.DataArray | str,
    ds: xr.Dataset | None,
    argument: str,
) -> xr.DataArray:
    """Resolve a DataArray or a variable name from a Dataset."""
    if isinstance(value, xr.DataArray):
        return value
    if ds is None:
        raise ValueError(f"{argument!r} is a variable name, so ds must be supplied")
    try:
        return ds[value]
    except KeyError as exc:
        raise KeyError(f"Dataset has no variable {value!r} for {argument!r}") from exc


def _threshold_magnitude(value: Any, target_units: str) -> float:
    """Convert a numeric or '<magnitude> <units>' threshold."""
    if isinstance(value, str):
        magnitude, source_units = value.strip().split(maxsplit=1)
        return float(
            ek_convert_units(
                float(magnitude),
                target_units=target_units,
                source_units=source_units,
            )
        )
    return float(value)


@format_handler()
def fusarium_head_blight_conducive_days_earthkit(
    tas: xr.DataArray | str = "tas",
    hurs: xr.DataArray | str = "hurs",
    pr: xr.DataArray | str = "pr",
    ds: xr.Dataset | Any = None,
    *,
    tas_lower: Any = "15 degC",
    tas_upper: Any = "30 degC",
    hurs_lower: Any = "90 %",
    pr_lower: Any = "0 mm",
    freq: str = "YS",
    **kwargs: Any,
) -> xr.DataArray:
    """Compute yearly FHB-conducive days without xclim."""
    tas = _resolve_dataarray(tas, ds, "tas")
    hurs = _resolve_dataarray(hurs, ds, "hurs")
    pr = _resolve_dataarray(pr, ds, "pr")

    tas_c = ek_convert_units(tas, target_units="degC")
    hurs_pct = ek_convert_units(hurs, target_units="%")
    pr_mm_per_day = ek_convert_units(pr, target_units="mm/day")

    lower_t = _threshold_magnitude(tas_lower, "degC")
    upper_t = _threshold_magnitude(tas_upper, "degC")
    lower_rh = _threshold_magnitude(hurs_lower, "%")
    lower_pr = _threshold_magnitude(pr_lower, "mm")

    conducive = (tas_c >= lower_t) & (tas_c <= upper_t) & (hurs_pct >= lower_rh) & (pr_mm_per_day > lower_pr)
    valid = tas.notnull() & hurs.notnull() & pr.notnull()
    conducive = conducive.where(valid)

    out = ekt.resample(conducive, frequency=freq, how="sum", how_kwargs={"skipna": False})
    out = out.rename("fhb_conducive_days")

    source_cell_methods = " ".join(
        f"{argument}: {data.attrs['cell_methods']}"
        for argument, data in (("tas", tas), ("hurs", hurs), ("pr", pr))
        if data.attrs.get("cell_methods")
    )
    description = (
        f"Number of days where {tas_lower} <= T <= {tas_upper}, relative humidity "
        f">= {hurs_lower}, and precipitation > {pr_lower}."
    ).capitalize()
    out.attrs = {
        "cell_methods": f"{source_cell_methods} time: sum over days".strip(),
        "units": "d",
        "long_name": "Fusarium head blight conducive days",
        "description": description,
    }
    return out

## 7. Compare the two approaches

Both implementations receive the same dataset and must produce identical yearly results and stable metadata. `DataArray.equals()` checks values, dimensions, and coordinates but intentionally ignores attributes. Therefore, the comparison also uses `DataArray.identical()` after removing xclim's timestamped `history` provenance attribute.

In [11]:
fhb_days_earthkit = fusarium_head_blight_conducive_days_earthkit(ds=ds)

np.testing.assert_array_equal(fhb_days_earthkit.values, [3, 4])
assert fhb_days_earthkit.equals(fhb_days)

# `equals` ignores attrs. `identical` also checks the name and all remaining attrs.
fhb_days_xclim_stable = fhb_days.copy()
fhb_days_xclim_stable.attrs.pop("history", None)
assert fhb_days_earthkit.identical(fhb_days_xclim_stable)

# With skipna=False passed to the sum, one missing daily input invalidates that year's count.
ds_with_missing = ds.copy(deep=True)
ds_with_missing["tas"].loc[{"time": "2020-06-10"}] = np.nan
fhb_days_with_missing = fusarium_head_blight_conducive_days_earthkit(ds=ds_with_missing)
assert np.isnan(fhb_days_with_missing.sel(time="2020").item())
assert fhb_days_with_missing.sel(time="2021").item() == 4

xr.Dataset({
    "xclim_backed": fhb_days,
    "earthkit_only": fhb_days_earthkit,
}).to_dataframe()

,xclim_backed,earthkit_only
time,,
2020-01-01,3.0,3.0
2021-01-01,4.0,4.0


The earthkit-only implementation is shorter and gives full control over the calculation. The xclim-backed implementation additionally supplies a formal indicator object, standardized parameter introspection, configurable missing-value policies, validation, and richer automatic metadata. The appropriate choice depends on whether the new indicator needs those catalogue-level behaviours or only a clear earthkit-compatible function.